In [1]:
%pip install pandas numpy "psycopg[binary]"

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------- ----------- 2.6/3.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 8.7 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   ---------------------------------------- 2/2 [psycopg]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import psycopg
import hashlib

from decimal import Decimal
from pathlib import Path



In [10]:
from datetime import datetime, timedelta

np.random.seed(42)

total_records = 100_000

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

file_path = data_dir / "sales_2026_10_28.csv"

df = pd.DataFrame({
    "order_id": np.arange(100001, 100001 + total_records),
    "customer_id": [
        f"C{n:05d}"
        for n in np.random.randint(1, 10001, total_records)
    ],
    "amount": np.round(
        np.random.uniform(100, 50000, total_records), 2
    ),
    "updated_at": [
        datetime(2026, 10, 1) + timedelta(minutes=int(n))
        for n in np.random.randint(0, 40000, total_records)
    ]
})

df.to_csv(file_path, index=False)

print("File:", file_path)
print("Total records:", len(df))
print("File size (MB):", round(file_path.stat().st_size / (1024**2), 2))

File: data\sales_2026_10_28.csv
Total records: 100000
File size (MB): 4.17


In [11]:
sample_df = pd.read_csv(file_path, nrows=5)

display(sample_df)

print("Columns:", sample_df.columns.tolist())

,order_id,customer_id,amount,updated_at
0,100001,C07271,7502.72,2026-10-07 04:14:00
1,100002,C00861,1919.00,2026-10-06 03:07:00
2,100003,C05391,22306.40,2026-10-11 01:31:00
3,100004,C05192,11696.04,2026-10-08 22:45:00
4,100005,C05735,46771.06,2026-10-04 17:54:00


Columns: ['order_id', 'customer_id', 'amount', 'updated_at']


In [6]:
from getpass import getpass
import psycopg

pg_password = getpass("PostgreSQL password: ")

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="sales_etl",
    user="postgres",
    password=pg_password
)

print("PostgreSQL connected successfully!")

PostgreSQL password:  ········


PostgreSQL connected successfully!


In [12]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = 'bronze'
          AND table_name = 'sales_raw'
    """)
    print(cur.fetchone())

('bronze', 'sales_raw')


In [13]:
import pandas as pd
import hashlib
from decimal import Decimal

file_path = "data/sales_2026_10_28.csv"
source_file = "sales_2026_10_28.csv"
batch_id = "sales_2026_10_28"
chunksize = 5000

insert_sql = """
INSERT INTO bronze.sales_raw (
    order_id,
    customer_id,
    amount,
    updated_at,
    source_file,
    batch_id,
    record_hash
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (source_file, record_hash) DO NOTHING
"""

total_read = 0
total_inserted = 0

try:
    with conn.cursor() as cur:
        for batch_no, chunk in enumerate(
            pd.read_csv(file_path, chunksize=chunksize),
            start=1
        ):
            records = []

            for row in chunk.itertuples(index=False):
                order_id = int(row.order_id)
                customer_id = str(row.customer_id)
                amount = Decimal(str(row.amount)).quantize(
                    Decimal("0.01")
                )
                updated_at = pd.Timestamp(
                    row.updated_at
                ).to_pydatetime()

                record = (
                    f"{order_id}|{customer_id}|"
                    f"{amount:.2f}|"
                    f"{updated_at.isoformat(sep=' ')}"
                )

                record_hash = hashlib.sha256(
                    record.encode("utf-8")
                ).hexdigest()

                records.append((
                    order_id,
                    customer_id,
                    amount,
                    updated_at,
                    source_file,
                    batch_id,
                    record_hash
                ))

            cur.executemany(insert_sql, records)
            inserted = cur.rowcount
            conn.commit()

            total_read += len(records)
            total_inserted += inserted

            print(
                f"Batch {batch_no}: "
                f"Read={len(records)}, Inserted={inserted}"
            )

except Exception:
    conn.rollback()
    raise

print("\nIngestion completed")
print("Total records read:", total_read)
print("Total inserted:", total_inserted)

Batch 1: Read=5000, Inserted=0
Batch 2: Read=5000, Inserted=0
Batch 3: Read=5000, Inserted=0
Batch 4: Read=5000, Inserted=0
Batch 5: Read=5000, Inserted=0
Batch 6: Read=5000, Inserted=0
Batch 7: Read=5000, Inserted=0
Batch 8: Read=5000, Inserted=0
Batch 9: Read=5000, Inserted=0
Batch 10: Read=5000, Inserted=0
Batch 11: Read=5000, Inserted=0
Batch 12: Read=5000, Inserted=0
Batch 13: Read=5000, Inserted=0
Batch 14: Read=5000, Inserted=0
Batch 15: Read=5000, Inserted=0
Batch 16: Read=5000, Inserted=0
Batch 17: Read=5000, Inserted=0
Batch 18: Read=5000, Inserted=0
Batch 19: Read=5000, Inserted=0
Batch 20: Read=5000, Inserted=0

Ingestion completed
Total records read: 100000
Total inserted: 0
